# Week 33, Day 3: Running mypy

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w033d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import re

def postcode_area(postcode: str) -> str:
    return re.match(r"([A-Z]+)\d", postcode).group(1)

print(postcode_area("LS1 4AP"), postcode_area("SW1A 1AA"))

> **Key idea.** Install mypy in the project's virtual environment and run `mypy file.py` or `mypy .`; exit code 0 means no problems. Each report line gives the file and line, a message and an error code in brackets; `reveal_type` shows what mypy believes. mypy is gradual: unknown values are `Any`, and unannotated function bodies aren't checked unless you ask (`--check-untyped-defs`). Raise strictness with options or `strict = true` in `pyproject.toml`, using `[[tool.mypy.overrides]]` per module, and stub packages or `ignore_missing_imports` for libraries without hints. Silence with `# type: ignore[code]` only when you must, never bare.

### Exercise: Reading reports with code

Write a NamedTuple `Problem` with fields `file`, `line` (an `int`), `message` and `code`, and `parse_report(text)`, which turns a mypy report into a list of `Problem`s, one per `error:` line, in order. Skip `note:` lines and the summary line. Then write `summary(problems)`: a dictionary from each error code to how many times it appears.

In [ ]:
import re
from typing import NamedTuple

___

def parse_report(text: str) -> list[Problem]:
    ___

def summary(problems: list[Problem]) -> dict[str, int]:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("REPORT = '''shop.py:11: error: \"Account\" has no attribute \"ballance\"; maybe \"balance\"?  [attr-defined]\nshop.py:22: error: Missing positional argument \"text\" in call to \"send\"  [call-arg]\nshop.py:23: error: Too many arguments for \"send\"  [call-arg]\nshop.py:32: error: Signature of \"area\" incompatible with supertype \"Shape\"  [override]\nshop.py:32: note:      Superclass:\nshop.py:32: note:          def area(self) -> float\nlib/util.py:7: error: Item \"None\" of \"str | None\" has no attribute \"upper\"  [union-attr]\nFound 5 errors in 2 files (checked 2 source files)'''\ndef test_parse():\n    \"parsing\"\n    P = ns[\"Problem\"]\n    ps = parse_report(REPORT)\n    assert len(ps) == 5 and ps[0] == P(\"shop.py\", 11, '\"Account\" has no attribute \"ballance\"; maybe \"balance\"?', \"attr-defined\")\n    assert ps[4] == P(\"lib/util.py\", 7, 'Item \"None\" of \"str | None\" has no attribute \"upper\"', \"union-attr\") and isinstance(ps[1].line, int)\n    assert summary(ps) == {\"attr-defined\": 1, \"call-arg\": 2, \"override\": 1, \"union-attr\": 1}\n    assert parse_report(\"Success: no issues found in 1 source file\") == [] and summary([]) == {}")

### Exercise: A mypy configuration

Write the text of a `pyproject.toml` in the string `CONFIG`, with a `[tool.mypy]` section that sets `python_version` to `"3.12"`, turns on `strict`, and turns on `warn_unreachable`; then an override section for the modules matching `"legacy.*"` that sets `ignore_errors`; and another for the module `"fancy_receipts"` that sets `ignore_missing_imports`. The checks read it with `tomllib`, Python's TOML reader.

In [ ]:
CONFIG = """
___
"""

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_config():\n    \"the configuration\"\n    import tomllib\n    data = tomllib.loads(CONFIG)[\"tool\"][\"mypy\"]\n    assert data[\"python_version\"] == \"3.12\" and data[\"strict\"] is True and data[\"warn_unreachable\"] is True\n    over = {o[\"module\"]: o for o in data[\"overrides\"]}\n    assert over[\"legacy.*\"][\"ignore_errors\"] is True and over[\"fancy_receipts\"][\"ignore_missing_imports\"] is True\n    assert len(data[\"overrides\"]) == 2")

### Exercise: Ready for --strict

Under `--strict`, mypy reports `Function is missing a type annotation  [no-untyped-def]` for every function here. Add complete hints so that every function has every parameter (except `self`) and its return annotated, without changing the behaviour: names are text, amounts whole numbers of pence, `stock` maps item names to quantities, and `restock` returns nothing.

In [ ]:
class Shop:
    def __init__(self, name, stock=None):
        self.name = name
        self.stock = dict(stock or {})

    def restock(self, item, qty):
        self.stock[item] = self.stock.get(item, 0) + qty

    def in_stock(self, item):
        return self.stock.get(item, 0) > 0

    def items(self):
        return sorted(self.stock)

def price_with_vat(pence, rate=20):
    return pence + pence * rate // 100

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hints():\n    \"complete hints\"\n    import inspect\n    from typing import get_type_hints as h\n    S = ns[\"Shop\"]\n    for f in [S.__init__, S.restock, S.in_stock, S.items, price_with_vat]:\n        hints = h(f)\n        names = [n for n in inspect.signature(f).parameters if n != \"self\"]\n        missing = [n for n in names + [\"return\"] if n not in hints]\n        assert not missing, f\"{f.__name__} is missing hints for {missing}\"\n    assert h(S.__init__) == {\"name\": str, \"stock\": dict[str, int] | None, \"return\": type(None)}, f\"__init__: {h(S.__init__)}\"\n    assert h(S.restock) == {\"item\": str, \"qty\": int, \"return\": type(None)} and h(S.in_stock) == {\"item\": str, \"return\": bool}\n    assert h(S.items) == {\"return\": list[str]} and h(price_with_vat) == {\"pence\": int, \"rate\": int, \"return\": int}\n    s = S(\"Corner\", {\"rice\": 2})\n    s.restock(\"oil\", 3)\n    assert s.items() == [\"oil\", \"rice\"] and s.in_stock(\"oil\") and not s.in_stock(\"salt\") and price_with_vat(1000) == 1200")

### Exercise: An override that fits

mypy reports:

```text
rugs.py:12: error: Signature of "area" incompatible with supertype "Shape"  [override]
rugs.py:12: note:      Superclass:
rugs.py:12: note:          def area(self) -> float
rugs.py:12: note:      Subclass:
rugs.py:12: note:          def area(self, units: str) -> float
```

and `total_area` crashes on the rug. Fix `Rug.area` so it fits its parent: it should still accept `units` (`"m"` or `"cm"`), but as an optional argument defaulting to `"m"`. In centimetres, the area is 10000 times bigger.

In [ ]:
class Shape:
    def area(self) -> float:
        return 0.0

class Square(Shape):
    def __init__(self, side: float) -> None:
        self.side = side

    def area(self) -> float:
        return self.side ** 2

class Rug(Shape):
    def __init__(self, width: float, length: float) -> None:
        self.width = width
        self.length = length

    def area(self, units: str) -> float:
        a = self.width * self.length
        return a * 10000 if units == "cm" else a

def total_area(shapes: list[Shape]) -> float:
    return sum(s.area() for s in shapes)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_override():\n    \"fits the parent\"\n    import inspect\n    R, S = ns[\"Rug\"], ns[\"Square\"]\n    assert total_area([S(2), R(1.5, 2)]) == 7.0 and R(1.5, 2).area(\"cm\") == 30000.0 and R(1, 1).area() == 1\n    assert inspect.signature(R.area).parameters[\"units\"].default == \"m\"")

### Bug hunt: The attribute that isn't there

mypy reports `error: "Account" has no attribute "ballance"; maybe "balance"?  [attr-defined]` on the `return` line of `report`, and running it crashes with `AttributeError`. Fix it so it prints `Ada: 500`.

In [ ]:
class Account:
    def __init__(self, owner: str, balance: int = 0) -> None:
        self.owner = owner
        self.balance = balance

def report(account: Account) -> str:
    return f"{account.owner}: {account.ballance}"

print(report(Account("Ada", 500)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the report\"\n    expect_output(\"Ada: 500\")")

### Bug hunt: No match

mypy reports `error: Item "None" of "Match[str] | None" has no attribute "group"  [union-attr]`, and the program crashes on the postcode with a space in front. Fix `postcode_area` so it strips spaces and makes the postcode upper case first, and raises `ValueError` (not `AttributeError`) when there's still no match. It should print `LS`, `SW` and then `bad postcode`.

In [ ]:
import re

def postcode_area(postcode: str) -> str:
    return re.match(r"([A-Z]+)\d", postcode).group(1)

for p in ["LS1 4AP", " sw1a 1aa", "12345"]:
    try:
        print(postcode_area(p))
    except ValueError:
        print("bad postcode")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"areas\"\n    expect_output(\"LS\\nSW\\nbad postcode\")\n    assert \"type: ignore\" not in source")

### Bug hunt: A missing argument

mypy reports `error: Missing positional argument "text" in call to "send"  [call-arg]`, and the program crashes with a `TypeError`. The reminder should be sent to Tunde with the text `Training at 6`, marked urgent. Fix the call so it prints `To Tunde (urgent): Training at 6`.

In [ ]:
def send(to: str, text: str, urgent: bool = False) -> None:
    flag = " (urgent)" if urgent else ""
    print(f"To {to}{flag}: {text}")

send("Tunde", urgent=True)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sent\"\n    expect_output(\"To Tunde (urgent): Training at 6\")")

### Puzzle: Checking a match

Arrange the lines into a function that mypy accepts, which returns the year in a date like `"2026-11-03"`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

For each snippet, would default mypy (no extra options) report an error? Answer `True` or `False`.

In [ ]:
# a:
#     def f(x: int) -> str:
#         return x
# b:
#     def g(x):
#         return x + "1" + 2
# c:
#     n: int = True
# d:
#     def h(name: str | None) -> int:
#         return len(name)
# e:
#     def k(v: float) -> float:
#         return v
#     k(3)

a = ___
b = ___
c = ___
d = ___
e = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (a, b, c) == (True, False, False), \"a to c\"\n    assert (d, e) == (True, False)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tiny type checker

Write `check(source)`, a very small type checker in the spirit of mypy, using Python's `ast` module (Week 22, Day 4). It reads source code without running it and returns a list of problems as strings, sorted by line number (and, on the same line, in the order below). It only understands functions whose hints are simple names (`int`, `float`, `str`, `bool`), and only checks **literal** values (numbers, strings, `True`, `False`):

- **Calls** to a function defined in the same source, by name, with only positional arguments:
  - if the number of arguments is fewer than the parameters without defaults, or more than all the parameters: `line 9: "send" takes 2 to 3 arguments, got 4` (or `takes 2 arguments` when the two numbers are equal);
  - for each literal argument whose type doesn't fit its parameter's hint: `line 9: argument 1 to "send" is str, expected int`.
- **Returns** inside such a function, when the returned value is a literal that doesn't fit the return hint: `line 4: "parse_age" returns str, expected int`.

A value fits a hint if its type is the hint, with mypy's two conveniences: an `int` fits `float`, and a `bool` fits `int` (and so `float`). Parameters and returns without hints aren't checked.

In [ ]:
import ast

def check(source: str) -> list[str]:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("CODE = '''\ndef parse_age(text: str) -> int:\n    if text == \"\":\n        return \"unknown\"\n    return 0\n\ndef send(to: str, amount: int, urgent: bool = False) -> None:\n    return None\n\nsend(\"Ada\", \"500\")\nsend(\"Ada\", 500, True, \"now\")\nsend(42, 1)\nsend(\"Bo\")\n\ndef area(w: float, h) -> float:\n    return 1\n\narea(2, \"x\")\narea(True, 3.5)\nprint(\"not ours\", 1)\n'''\ndef test_check():\n    \"problems\"\n    got = check(CODE)\n    want = [\n        'line 4: \"parse_age\" returns str, expected int',\n        'line 10: argument 2 to \"send\" is str, expected int',\n        'line 11: \"send\" takes 2 to 3 arguments, got 4',\n        'line 12: argument 1 to \"send\" is int, expected str',\n        'line 13: \"send\" takes 2 to 3 arguments, got 1',\n    ]\n    assert got == want, \"got:\\n\" + \"\\n\".join(got)\n    assert check(\"def f(a: int) -> int:\\n    return a\\n\\nf(1)\\n\") == []\n    assert check(\"def f(a: int, b: int) -> None:\\n    pass\\n\\nf(1)\\n\") == ['line 4: \"f\" takes 2 arguments, got 1']\n    assert check(\"def f(x: int) -> str:\\n    return True\\n\") == ['line 2: \"f\" returns bool, expected str']")

## Warm-up

It works for both, and would pass any test written with real postcodes. mypy, without running anything, reports:

```text
shop.py:4: error: Item "None" of "Match[str] | None" has no attribute "group"  [union-attr]
```

`re.match` returns `None` when the text doesn't match, which happens the day someone types a postcode in lower case or with a space at the front, and the program crashes. mypy knows `re.match`'s return type from the standard library's hints, and noticed the case nobody tested. That's what a type checker is for.

## Setting up and running

mypy is a separate tool, installed into each project's virtual environment (Week 22, Day 2) like any other package. On your own computer, in the project folder:

```bash
python -m venv .venv
source .venv/bin/activate        # on Windows: .venv\Scripts\activate
pip install mypy
mypy shop.py                     # check one file (and the modules it imports)
mypy .                           # check every file in the project
```

When everything fits, it says so and finishes with exit code 0 (Week 21: the code a program gives the shell, where 0 means success):

```text
Success: no issues found in 1 source file
```

Otherwise it lists each problem and finishes with exit code 1, which is what lets automatic checks (before each commit, or on every pull request) stop bad code getting in.

In VS Code, the **Mypy Type Checker** extension runs it as you work and underlines problems, just as Ruff did in Week 22. Many editors also run a type checker of their own as you type (VS Code's Python extension uses one called Pylance). Other checkers such as Pyright work the same way and read the same hints; this course uses mypy because it's the original and the most common.

mypy can't run in your browser on this site. Throughout this week, when you see a mypy report, it's real output from running mypy on that code.

## Reading the report

Every line of a report has the same parts:

```text
shop.py:11: error: "Account" has no attribute "ballance"; maybe "balance"?  [attr-defined]
```

| Part | Here |
|---|---|
| file and line | `shop.py:11` |
| severity | `error` (or `note`, which adds detail to the error before it) |
| message | `"Account" has no attribute "ballance"; maybe "balance"?` |
| error code | `[attr-defined]`, the kind of problem, used for searching the documentation and for silencing |

Here's a file with five common mistakes, and mypy's full report:

```python
class Account:                                       # line 4
    def __init__(self, owner: str, balance: int = 0) -> None:
        self.owner = owner
        self.balance = balance


def report(account: Account) -> str:
    return f"{account.owner}: {account.ballance}"    # line 11


def send(to: str, text: str, urgent: bool = False) -> None:
    print(to, text, urgent)


send("Ada")                                          # line 22
send("Ada", "hi", True, "now")                       # line 23


class Shape:
    def area(self) -> float:
        return 0.0


class Rug(Shape):
    def area(self, units: str) -> float:             # line 32
        return 6.0
```

```text
shop.py:11: error: "Account" has no attribute "ballance"; maybe "balance"?  [attr-defined]
shop.py:22: error: Missing positional argument "text" in call to "send"  [call-arg]
shop.py:23: error: Too many arguments for "send"  [call-arg]
shop.py:32: error: Signature of "area" incompatible with supertype "Shape"  [override]
shop.py:32: note:      Superclass:
shop.py:32: note:          def area(self) -> float
shop.py:32: note:      Subclass:
shop.py:32: note:          def area(self, units: str) -> float
Found 4 errors in 1 file (checked 1 source file)
```

The `override` error is Week 30, Day 3's rule ("a child must work wherever its parent does"), checked automatically: code that calls `shape.area()` on a `Rug` would crash.

The codes you'll meet most:

| Code | Usually means |
|---|---|
| `arg-type` | an argument of the wrong type |
| `call-arg` | too many or too few arguments, or an unknown keyword |
| `return-value` | returning the wrong type |
| `return` | a path through the function that doesn't return |
| `assignment` | putting the wrong type into an annotated variable |
| `attr-defined` | an attribute or method that doesn't exist (often a typo) |
| `union-attr` | using a value that might be `None` (or another type) as if it couldn't be |
| `operator` | an operator that doesn't work on those types |
| `var-annotated` | an empty collection that needs a hint |
| `override` | a child's method that doesn't fit its parent's |
| `import-not-found` | a module mypy can't find, or that has no hints |

When you're not sure what type mypy thinks something has, ask it: `reveal_type(expr)` in your code makes mypy print the type as a note (it's understood by mypy only; delete it before running the program):

```text
shop.py:41: note: Revealed type is "int"
shop.py:42: note: Revealed type is "def (postcode: str) -> str"
```

## Gradual typing

You don't have to annotate a whole project at once. mypy is built for **gradual typing**: hints can be added a bit at a time, and code without hints is treated generously.

- A value whose type mypy doesn't know has type `Any`, which can be used in any way without complaint (Day 2).
- By default, mypy **doesn't check the body of a function that has no hints at all**. This function has an obvious bug, and mypy reports nothing:

```python
def price_label(pence):
    return "£" + 100
```

```text
Success: no issues found in 1 source file
```

Add one hint (say, `pence: int`), or run mypy with `--check-untyped-defs`, and the body is checked:

```text
labels.py:2: error: Unsupported operand types for + ("str" and "int")  [operator]
```

That default lets an old project start using mypy without drowning in reports. The usual path is: run mypy on the project as it is, fix what it finds, then annotate the most important modules first (the ones other code depends on), and turn up the strictness as coverage grows.

## How strict?

mypy's options make it stricter. The most useful:

| Option | Effect |
|---|---|
| `--check-untyped-defs` | check the bodies of unannotated functions too |
| `--disallow-untyped-defs` | report every function without complete hints |
| `--warn-return-any` | report returning an `Any` from a function with a precise return hint |
| `--strict` | turn on all the strictness options together |

Under `--strict`, missing hints are errors, and so are silencing comments that aren't needed any more:

```python
def double(x):
    return x * 2

def label(name: str):
    return name.title()

n = int("3")  # type: ignore
```

```text
loose.py:1: error: Function is missing a type annotation  [no-untyped-def]
loose.py:4: error: Function is missing a return type annotation  [no-untyped-def]
loose.py:7: error: Unused "type: ignore" comment  [unused-ignore]
Found 3 errors in 1 file (checked 1 source file)
```

Rather than typing options every time, put them in `pyproject.toml` at the top of the project (Week 22 put Ruff's settings there too). `[[tool.mypy.overrides]]` sections change settings for some modules only, which is how a project can be strict for new code while older modules catch up:

```toml
[tool.mypy]
python_version = "3.12"
strict = true

[[tool.mypy.overrides]]
module = "legacy.*"
ignore_errors = true

[[tool.mypy.overrides]]
module = "fancy_receipts"
ignore_missing_imports = true
```

That last section deals with a third-party package that has no hints. mypy reports such packages like this:

```text
shop.py:1: error: Cannot find implementation or library stub for module named "fancy_receipts"  [import-not-found]
```

Many popular packages ship hints, and for others there are separate "stub" packages with just the hints, usually named `types-` followed by the package's name (`pip install types-requests`). If there are none, `ignore_missing_imports` for that one module tells mypy to treat it as `Any`.

## Silencing, carefully

Sometimes mypy is wrong, or the code is right in a way mypy can't follow. You can silence one line with a comment, naming the error code:

```python
value = legacy.compute()  # type: ignore[attr-defined]
```

Always give the code in brackets: a bare `# type: ignore` hides **every** error on that line, including ones you haven't made yet. With `warn_unused_ignores` (part of `strict`), mypy also tells you when a silencing comment is no longer needed.

But silencing should be rare. Most reports are real bugs, and the fix belongs in the code: check for `None`, correct the hint, rename the attribute. If you find yourself silencing often, the hints are probably wrong, not the checker.

## Practice

## Quiz

## Daily challenge

Real type checkers do this for every expression, not just literals, by working out the type of each value as it flows through the program. That's the same idea at a much larger scale.

## Before you go

If you have a project of your own (from Week 25, perhaps), install mypy in its virtual environment and run it. Fix what it finds, then try `--check-untyped-defs`. Tomorrow: generics, `TypeVar` and `Callable`, for functions and classes that work with any type while keeping track of which.